# Implement a simple Article SubGraph

In [1]:
%cd ..

c:\Users\Tuan Kiet\Desktop\Workspace\VLSP_2025_SIGMA


In [13]:
import json
import logging
from pydantic import BaseModel, Field
from typing import  Annotated, List, Dict

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage

from src.services.llm_service import AzureChatModel
from src.utils import format_choices, extract_json_from_deepseek_response, get_article_text
from src.prompts import CHECK_ARTICLE_RELEVANCY_PROMPT

llm = AzureChatModel().client

# --- Logging setup ---
logger = logging.getLogger("ArticleAgent")
logging.basicConfig(level=logging.INFO)

class Article(BaseModel):
    title: str
    text: str

    def __str__(self):
        """
        Returns a human-readable string representation of the article,
        showing the title and a truncated version of the text.
        """
        preview = self.text[:50] + "..." if len(self.text) > 50 else self.text
        return f"Title: {self.title}\nText: {preview}"

    def __repr__(self):
        """
        Returns a developer-friendly string representation of the article,
        useful for debugging and logging.
        """
        preview = self.text[:50] + "..." if len(self.text) > 50 else self.text
        return f"Article(title={self.title!r}, text={preview!r})"

        

# --- Define State ---
class ArticleState(BaseModel):
    question: str
    choices: Dict
    image_analysis: str
    articles: List[Article]
    current_index: int = Field(default=0, description="Index of the article currently being analyzed")
    relevant_articles: List[Article] = Field(default_factory=list, description="List of relevant articles")
    messages: Annotated[List[BaseMessage], Field(default_factory=lambda: [
        HumanMessage(content="Bắt đầu phân tích các điều luật liên quan")
    ])]

# --- Helper functions ---
def no_more_articles(state: ArticleState) -> bool:
    return state.current_index >= len(state.articles)

def article_is_relevant(state: ArticleState) -> bool:
    try:
        last_msg = state.messages[-1].content
        parsed = json.loads(last_msg)
        return parsed.get("relevant", False) is True
    except Exception as e:
        logger.warning(f"Failed to parse relevance from LLM response: {e}")
        return False

# --- Agent Tool ---
class AnalyzeArticleInput(BaseModel):
    question: str
    choices: List[str]
    image_analysis: str
    article: str

# --- Nodes ---
def analyze_article(state: ArticleState):
    logger.info(f"[analyze_article] Analyzing article at index {state.current_index}")

    prompt = CHECK_ARTICLE_RELEVANCY_PROMPT.format(
        question=state.question,
        choices=format_choices(state.choices),
        article=get_article_text([article.model_dump() for article in state.articles], state.current_index)
    )

    response = llm.invoke([HumanMessage(content=prompt)])

    response.content = extract_json_from_deepseek_response(response) 

    state.messages.append(response)
    return state

def check_next_article(state: ArticleState):
    if no_more_articles(state):
        logger.info("[check_next_article] No more articles. Ending subgraph.")
    else:
        logger.info(f"[check_next_article] Moving to article index {state.current_index}")
    
    return state

def save_article(state: ArticleState):
    logger.info(f"[save_article] Saving relevant article at index {state.current_index}")
    last_msg = state.messages[-1].content

    state.relevant_articles.append(state.articles[state.current_index])
    state.current_index += 1
    return state

def skip_article(state: ArticleState):
    logger.info(f"[skip_article] Skipping article at index {state.current_index}")
    state.current_index += 1
    return state

# --- Build Subgraph ---
builder = StateGraph(ArticleState)

builder.add_node("check_next_article", check_next_article)
builder.add_node("analyze_article", analyze_article)
builder.add_node("save_article", save_article)
builder.add_node("skip_article", skip_article)

builder.set_entry_point("check_next_article")

builder.add_edge("save_article", "check_next_article")
builder.add_edge("skip_article", "check_next_article")

builder.add_conditional_edges(
    "check_next_article",
    lambda state: "END" if no_more_articles(state) else "analyze_article",
    {
        "END": END,
        "analyze_article": "analyze_article"
    }
)

builder.add_conditional_edges(
    "analyze_article",
    lambda state: "save_article" if article_is_relevant(state) else "skip_article",
    {
        "save_article": "save_article",
        "skip_article": "skip_article"
    }
)

article_subgraph = builder.compile()


In [1]:
from langchain_core.messages import HumanMessage
import logging

# Giảm mức log của http_logging_policy xuống WARNING
logging.getLogger("azure.core.pipeline.policies.http_logging_policy").setLevel(logging.WARNING)


example_state = {
    "question": "Biển báo yêu cầu tốc độ tối đa là bao nhiêu?",
    "choices": {"A": "60 km/h", "B": "80 km/h", "C": "100 km/h", "D": "Không có giới hạn"},
    "image_analysis": "Biển báo là biển số 127c, cho thấy giới hạn tốc độ 60 km/h cho xe ô tô.",
    "articles": [
        {
            'id': '22',
            'text': '22.1. Biển báo cấm có mã P (cấm) và DP (hết cấm) với tên các biển như sau:\n- Biển số P.101: Đường cấm;\n- Biển số P.102: Cấm đi ngược chiều;\n- Biển số P.103a: Cấm xe ô tô;\n- Biển số P.103(b,c): Cấm xe ô tô rẽ trái; Cấm xe ôtô rẽ phải;\n- Biển số P.104: Cấm xe máy;\n- Biển số P.105: Cấm xe ô tô và xe máy;\n- Biển số P.106(a,b): Cấm xe ô tô tải;\n- Biển số P.106c: Cấm các xe chở hàng nguy hiểm;\n- Biển số P.107: Cấm xe ô tô khách và xe ô tô tải;\n- Biển số P.107a: Cấm xe ô tô khách;\n- Biển số P.107b: Cấm xe ô tô taxi;\n- Biển số P.108: Cấm xe kéo rơ-moóc;\n- Biển số P.108a: Cấm xe sơ-mi rơ-moóc;\n- Biển số P.109: Cấm máy kéo;\n- Biển số P.110a: Cấm xe đạp;\n- Biển số P.110b: Cấm xe đạp thồ;\n- Biển số P.111a: Cấm xe gắn máy;\n- Biển số P.111(b) hoặc (c): Cấm xe ba bánh loại có động cơ (xe lam, xích lô máy);\n- Biển số P.111d: Cấm xe ba bánh loại không có động cơ (xích lô);\n- Biển số P.112: Cấm người đi bộ;\n- Biển số P.113: Cấm xe người kéo, đẩy;\n- Biển số P.114: Cấm xe vật nuôi kéo;\n- Biển số P.115: Hạn chế trọng tải toàn bộ xe;\n- Biển số P.116: Hạn chế tải trọng trên trục xe;\n- Biển số P.117: Hạn chế chiều cao;\n- Biển số P.118: Hạn chế chiều ngang xe;\n- Biển số P.119: Hạn chế chiều dài xe;\n- Biển số P.120: Hạn chế chiều dài xe cơ giới kéo theo rơ-moóc hoặc sơ-mi rơ - moóc;\n- Biển số P.121: Cự ly tối thiểu giữa hai xe;\n- Biển số P.123(a,b): Cấm rẽ trái; Cấm rẽ phải;\n- Biển số P.124(a,b): Cấm quay đầu xe; cấm ôtô quay đầu xe;\n- Biển số P.124(c,d): Cấm rẽ trái và quay đầu xe; Cấm rẽ phải và quay đầu xe;\n- Biển số P.124(e,f): Cấm ô tô rẽ trái và quay đầu xe; Cấm ô tô rẽ phải và quay đầu xe;\n- Biển số P.125: Cấm vượt;\n- Biển số P.126: Cấm xe ôtô tải vượt;\n- Biển số P.127: Tốc độ tối đa cho phép;\n- Biển số P.127a: Tốc độ tối đa cho phép về ban đêm;\n- Biển số P.127b: Biển ghép tốc độ tối đa cho phép trên từng làn đường;\n- Biển số P.127c: Biển ghép tốc độ tối đa cho phép theo phương tiện, trên từng làn đường;\n- Biển số DP.127: Biển hết tốc độ tối đa cho phép theo biển ghép;\n- Biển số P.128: Cấm sử dụng còi;\n- Biển số P.129: Kiểm tra;\n- Biển số P.130: Cấm dừng xe và đỗ xe;\n- Biển số P.131(a,b,c): Cấm đỗ xe;\n- Biển số P.132: Nhường đường cho xe cơ giới đi ngược chiều qua đường hẹp;\n- Biển số DP.133: Hết cấm vượt;\n- Biển số DP.134: Hết tốc độ tối đa cho phép;\n- Biển số DP.135: Hết tất cả các lệnh cấm;\n- Biển số P.136: Cấm đi thẳng;\n- Biển số P.137: Cấm rẽ trái, rẽ phải;\n- Biển số P.138: Cấm đi thẳng, rẽ trái;\n- Biển số P.139: Cấm đi thẳng, rẽ phải;\n- Biển số P.140: Cấm xe công nông và các loại xe tương tự.\n22.2. Ý nghĩa sử dụng của từng biển được giải thích chi tiết ở Phụ lục B.\n',
            'title': 'Ý nghĩa sử dụng các biển báo cấm'
        }
    ],
    "current_index": 0,
    "relevant_articles": [],
    "messages": [HumanMessage(content="Bắt đầu phân tích các điều luật liên quan")]
}

result_state = article_subgraph.invoke(ArticleState(**example_state))

import pprint
pprint.pprint(result_state)


NameError: name 'article_subgraph' is not defined

In [15]:
print(ArticleState(**result_state))

question='Biển báo yêu cầu tốc độ tối đa là bao nhiêu?' choices={'A': '60 km/h', 'B': '80 km/h', 'C': '100 km/h', 'D': 'Không có giới hạn'} image_analysis='Biển báo là biển số 127c, cho thấy giới hạn tốc độ 60 km/h cho xe ô tô.' articles=[Article(title='Ý nghĩa sử dụng các biển báo cấm', text='22.1. Biển báo cấm có mã P (cấm) và DP (hết cấm) v...')] current_index=1 relevant_articles=[Article(title='Ý nghĩa sử dụng các biển báo cấm', text='22.1. Biển báo cấm có mã P (cấm) và DP (hết cấm) v...')] messages=[HumanMessage(content='Bắt đầu phân tích các điều luật liên quan', additional_kwargs={}, response_metadata={}), AIMessage(content='{\n  "reason": "Điều luật liệt kê biển số P.127 là \'Tốc độ tối đa cho phép\', đây chính là biển báo yêu cầu tốc độ tối đa được đề cập trong câu hỏi. Thông tin này trực tiếp giúp xác định loại biển báo liên quan đến câu hỏi về giới hạn tốc độ.",\n  "relevant": true\n}', additional_kwargs={}, response_metadata={'model': 'DeepSeek-r1-0528', 'token_usage': {'in

In [17]:
article = {'id': '22',
  'text': '22.1. Biển báo cấm có mã P (cấm) và DP (hết cấm) với tên các biển như sau:\n- Biển số P.101: Đường cấm;\n- Biển số P.102: Cấm đi ngược chiều;\n- Biển số P.103a: Cấm xe ô tô;\n- Biển số P.103(b,c): Cấm xe ô tô rẽ trái; Cấm xe ôtô rẽ phải;\n- Biển số P.104: Cấm xe máy;\n- Biển số P.105: Cấm xe ô tô và xe máy;\n- Biển số P.106(a,b): Cấm xe ô tô tải;\n- Biển số P.106c: Cấm các xe chở hàng nguy hiểm;\n- Biển số P.107: Cấm xe ô tô khách và xe ô tô tải;\n- Biển số P.107a: Cấm xe ô tô khách;\n- Biển số P.107b: Cấm xe ô tô taxi;\n- Biển số P.108: Cấm xe kéo rơ-moóc;\n- Biển số P.108a: Cấm xe sơ-mi rơ-moóc;\n- Biển số P.109: Cấm máy kéo;\n- Biển số P.110a: Cấm xe đạp;\n- Biển số P.110b: Cấm xe đạp thồ;\n- Biển số P.111a: Cấm xe gắn máy;\n- Biển số P.111(b) hoặc (c): Cấm xe ba bánh loại có động cơ (xe lam, xích lô máy);\n- Biển số P.111d: Cấm xe ba bánh loại không có động cơ (xích lô);\n- Biển số P.112: Cấm người đi bộ;\n- Biển số P.113: Cấm xe người kéo, đẩy;\n- Biển số P.114: Cấm xe vật nuôi kéo;\n- Biển số P.115: Hạn chế trọng tải toàn bộ xe;\n- Biển số P.116: Hạn chế tải trọng trên trục xe;\n- Biển số P.117: Hạn chế chiều cao;\n- Biển số P.118: Hạn chế chiều ngang xe;\n- Biển số P.119: Hạn chế chiều dài xe;\n- Biển số P.120: Hạn chế chiều dài xe cơ giới kéo theo rơ-moóc hoặc sơ-mi rơ - moóc;\n- Biển số P.121: Cự ly tối thiểu giữa hai xe;\n- Biển số P.123(a,b): Cấm rẽ trái; Cấm rẽ phải;\n- Biển số P.124(a,b): Cấm quay đầu xe; cấm ôtô quay đầu xe;\n- Biển số P.124(c,d): Cấm rẽ trái và quay đầu xe; Cấm rẽ phải và quay đầu xe;\n- Biển số P.124(e,f): Cấm ô tô rẽ trái và quay đầu xe; Cấm ô tô rẽ phải và quay đầu xe;\n- Biển số P.125: Cấm vượt;\n- Biển số P.126: Cấm xe ôtô tải vượt;\n- Biển số P.127: Tốc độ tối đa cho phép;\n- Biển số P.127a: Tốc độ tối đa cho phép về ban đêm;\n- Biển số P.127b: Biển ghép tốc độ tối đa cho phép trên từng làn đường;\n- Biển số P.127c: Biển ghép tốc độ tối đa cho phép theo phương tiện, trên từng làn đường;\n- Biển số DP.127: Biển hết tốc độ tối đa cho phép theo biển ghép;\n- Biển số P.128: Cấm sử dụng còi;\n- Biển số P.129: Kiểm tra;\n- Biển số P.130: Cấm dừng xe và đỗ xe;\n- Biển số P.131(a,b,c): Cấm đỗ xe;\n- Biển số P.132: Nhường đường cho xe cơ giới đi ngược chiều qua đường hẹp;\n- Biển số DP.133: Hết cấm vượt;\n- Biển số DP.134: Hết tốc độ tối đa cho phép;\n- Biển số DP.135: Hết tất cả các lệnh cấm;\n- Biển số P.136: Cấm đi thẳng;\n- Biển số P.137: Cấm rẽ trái, rẽ phải;\n- Biển số P.138: Cấm đi thẳng, rẽ trái;\n- Biển số P.139: Cấm đi thẳng, rẽ phải;\n- Biển số P.140: Cấm xe công nông và các loại xe tương tự.\n22.2. Ý nghĩa sử dụng của từng biển được giải thích chi tiết ở Phụ lục B.\n',
  'title': 'Ý nghĩa sử dụng các biển báo cấm'}

In [2]:
from langchain_core.messages import HumanMessage
import json
state = {
    "question": "Biển báo yêu cầu tốc độ tối đa là bao nhiêu?",
    "choices": {
            "A": "Từ 6:30 đến 8:00 và từ 16:30 đến 18:30; ngoài các khoảng thời gian này không được phép lưu thông.",
            "B": "Từ 6:30 đến 8:00 và từ 16:30 đến 18:30; ngoài các khoảng thời gian này được phép lưu thông.",
            "C": "Cấm lưu thông cả ngày.",
            "D": "D. Không cấm xe khách trên 29 chỗ lưu thông."
        },
    "image_analysis": "Biển báo là biển số 127c, cho thấy giới hạn tốc độ 60 km/h cho xe ô tô.",
    "articles": [
        {
            'id': '22',
            'text': '22.1. Biển báo cấm có mã P (cấm) và DP (hết cấm) với tên các biển như sau:\n- Biển số P.101: Đường cấm;\n- Biển số P.102: Cấm đi ngược chiều;\n- Biển số P.103a: Cấm xe ô tô;\n- Biển số P.103(b,c): Cấm xe ô tô rẽ trái; Cấm xe ôtô rẽ phải;\n- Biển số P.104: Cấm xe máy;\n- Biển số P.105: Cấm xe ô tô và xe máy;\n- Biển số P.106(a,b): Cấm xe ô tô tải;\n- Biển số P.106c: Cấm các xe chở hàng nguy hiểm;\n- Biển số P.107: Cấm xe ô tô khách và xe ô tô tải;\n- Biển số P.107a: Cấm xe ô tô khách;\n- Biển số P.107b: Cấm xe ô tô taxi;\n- Biển số P.108: Cấm xe kéo rơ-moóc;\n- Biển số P.108a: Cấm xe sơ-mi rơ-moóc;\n- Biển số P.109: Cấm máy kéo;\n- Biển số P.110a: Cấm xe đạp;\n- Biển số P.110b: Cấm xe đạp thồ;\n- Biển số P.111a: Cấm xe gắn máy;\n- Biển số P.111(b) hoặc (c): Cấm xe ba bánh loại có động cơ (xe lam, xích lô máy);\n- Biển số P.111d: Cấm xe ba bánh loại không có động cơ (xích lô);\n- Biển số P.112: Cấm người đi bộ;\n- Biển số P.113: Cấm xe người kéo, đẩy;\n- Biển số P.114: Cấm xe vật nuôi kéo;\n- Biển số P.115: Hạn chế trọng tải toàn bộ xe;\n- Biển số P.116: Hạn chế tải trọng trên trục xe;\n- Biển số P.117: Hạn chế chiều cao;\n- Biển số P.118: Hạn chế chiều ngang xe;\n- Biển số P.119: Hạn chế chiều dài xe;\n- Biển số P.120: Hạn chế chiều dài xe cơ giới kéo theo rơ-moóc hoặc sơ-mi rơ - moóc;\n- Biển số P.121: Cự ly tối thiểu giữa hai xe;\n- Biển số P.123(a,b): Cấm rẽ trái; Cấm rẽ phải;\n- Biển số P.124(a,b): Cấm quay đầu xe; cấm ôtô quay đầu xe;\n- Biển số P.124(c,d): Cấm rẽ trái và quay đầu xe; Cấm rẽ phải và quay đầu xe;\n- Biển số P.124(e,f): Cấm ô tô rẽ trái và quay đầu xe; Cấm ô tô rẽ phải và quay đầu xe;\n- Biển số P.125: Cấm vượt;\n- Biển số P.126: Cấm xe ôtô tải vượt;\n- Biển số P.127: Tốc độ tối đa cho phép;\n- Biển số P.127a: Tốc độ tối đa cho phép về ban đêm;\n- Biển số P.127b: Biển ghép tốc độ tối đa cho phép trên từng làn đường;\n- Biển số P.127c: Biển ghép tốc độ tối đa cho phép theo phương tiện, trên từng làn đường;\n- Biển số DP.127: Biển hết tốc độ tối đa cho phép theo biển ghép;\n- Biển số P.128: Cấm sử dụng còi;\n- Biển số P.129: Kiểm tra;\n- Biển số P.130: Cấm dừng xe và đỗ xe;\n- Biển số P.131(a,b,c): Cấm đỗ xe;\n- Biển số P.132: Nhường đường cho xe cơ giới đi ngược chiều qua đường hẹp;\n- Biển số DP.133: Hết cấm vượt;\n- Biển số DP.134: Hết tốc độ tối đa cho phép;\n- Biển số DP.135: Hết tất cả các lệnh cấm;\n- Biển số P.136: Cấm đi thẳng;\n- Biển số P.137: Cấm rẽ trái, rẽ phải;\n- Biển số P.138: Cấm đi thẳng, rẽ trái;\n- Biển số P.139: Cấm đi thẳng, rẽ phải;\n- Biển số P.140: Cấm xe công nông và các loại xe tương tự.\n22.2. Ý nghĩa sử dụng của từng biển được giải thích chi tiết ở Phụ lục B.\n',
            'title': 'Ý nghĩa sử dụng các biển báo cấm'
        }
    ],
    "current_index": 0,
    "relevant_articles": [],
    "messages": [HumanMessage(content="Bắt đầu phân tích các điều luật liên quan")]
}

# response = llm.invoke(input=[HumanMessage(content=prompt)])

# print(response.content)

In [3]:
%cd ..

c:\Users\Tuan Kiet\Desktop\Workspace\VLSP_2025_SIGMA


In [7]:
from src.graphs import ArticleAnalysisSubgraph
from src.services.llm_service import AzureChatModel
from src.models import ArticleState

import json

llm = AzureChatModel().client

graph = ArticleAnalysisSubgraph(llm=llm)

input_state = ArticleState(**state)

# output_state = graph.run(input_state)

print(json.dumps(output_state.model_dump(), indent=4, ensure_ascii=False))

{
    "question": "Biển báo yêu cầu tốc độ tối đa là bao nhiêu?",
    "choices": {
        "A": "Từ 6:30 đến 8:00 và từ 16:30 đến 18:30; ngoài các khoảng thời gian này không được phép lưu thông.",
        "B": "Từ 6:30 đến 8:00 và từ 16:30 đến 18:30; ngoài các khoảng thời gian này được phép lưu thông.",
        "C": "Cấm lưu thông cả ngày.",
        "D": "D. Không cấm xe khách trên 29 chỗ lưu thông."
    },
    "image_analysis": "Biển báo là biển số 127c, cho thấy giới hạn tốc độ 60 km/h cho xe ô tô.",
    "articles": [
        {
            "title": "Ý nghĩa sử dụng các biển báo cấm",
            "text": "22.1. Biển báo cấm có mã P (cấm) và DP (hết cấm) với tên các biển như sau:\n- Biển số P.101: Đường cấm;\n- Biển số P.102: Cấm đi ngược chiều;\n- Biển số P.103a: Cấm xe ô tô;\n- Biển số P.103(b,c): Cấm xe ô tô rẽ trái; Cấm xe ôtô rẽ phải;\n- Biển số P.104: Cấm xe máy;\n- Biển số P.105: Cấm xe ô tô và xe máy;\n- Biển số P.106(a,b): Cấm xe ô tô tải;\n- Biển số P.106c: Cấm các xe chở hàn

In [25]:
print(prompt)

Bạn là một trợ lý pháp lý giao thông.
Dựa trên thông tin dưới đây, hãy đánh giá xem điều luật dưới đây có liên quan đến câu hỏi trắc nghiệm, các lựa chọn hay các phân tích về ảnh hay không.

**Câu hỏi:** Biển báo yêu cầu tốc độ tối đa là bao nhiêu?
**Lựa chọn:** ["A. 60 km/h", "B. 80 km/h", "C. 100 km/h", "D. Kh\u00f4ng c\u00f3 gi\u1edbi h\u1ea1n"]
**Phân tích hình ảnh:** Biển báo là biển số 127c, cho thấy giới hạn tốc độ 60 km/h cho xe ô tô.
**Điều luật:** {"id": "22", "text": "22.1. Bi\u1ec3n b\u00e1o c\u1ea5m c\u00f3 m\u00e3 P (c\u1ea5m) v\u00e0 DP (h\u1ebft c\u1ea5m) v\u1edbi t\u00ean c\u00e1c bi\u1ec3n nh\u01b0 sau:\n- Bi\u1ec3n s\u1ed1 P.101: \u0110\u01b0\u1eddng c\u1ea5m;\n- Bi\u1ec3n s\u1ed1 P.102: C\u1ea5m \u0111i ng\u01b0\u1ee3c chi\u1ec1u;\n- Bi\u1ec3n s\u1ed1 P.103a: C\u1ea5m xe \u00f4 t\u00f4;\n- Bi\u1ec3n s\u1ed1 P.103(b,c): C\u1ea5m xe \u00f4 t\u00f4 r\u1ebd tr\u00e1i; C\u1ea5m xe \u00f4t\u00f4 r\u1ebd ph\u1ea3i;\n- Bi\u1ec3n s\u1ed1 P.104: C\u1ea5m xe m\u00e1y;\n- Bi\u1

In [ ]:
from src.utils import load_json, load_law_db
from dotenv import load_dotenv

load_dotenv()
train_data = load_json("data/train_data/vlsp_2025_train.json")
train_data[0]
traffic_sign_law, traffic_order_safety_law = load_law_db()

c:\Users\Tuan Kiet\Desktop\Workspace\VLSP_2025_SIGMA


In [19]:
print(prompt)

('Bạn là một trợ lý pháp lý giao thông.\nDựa trên thông tin dưới đây, hãy đánh giá xem điều luật dưới đây có liên quan đến câu hỏi trắc nghiệm, các lựa chọn hay các phân tích về ảnh hay không.\n\n**Câu hỏi:** Biển báo yêu cầu tốc độ tối đa là bao nhiêu?\n**Lựa chọn:** ["A. 60 km/h", "B. 80 km/h", "C. 100 km/h", "D. Kh\\u00f4ng c\\u00f3 gi\\u1edbi h\\u1ea1n"]\n**Phân tích hình ảnh:** Biển báo là biển số 127c, cho thấy giới hạn tốc độ 60 km/h cho xe ô tô.\n**Điều luật:** {"id": "22", "text": "22.1. Bi\\u1ec3n b\\u00e1o c\\u1ea5m c\\u00f3 m\\u00e3 P (c\\u1ea5m) v\\u00e0 DP (h\\u1ebft c\\u1ea5m) v\\u1edbi t\\u00ean c\\u00e1c bi\\u1ec3n nh\\u01b0 sau:\\n- Bi\\u1ec3n s\\u1ed1 P.101: \\u0110\\u01b0\\u1eddng c\\u1ea5m;\\n- Bi\\u1ec3n s\\u1ed1 P.102: C\\u1ea5m \\u0111i ng\\u01b0\\u1ee3c chi\\u1ec1u;\\n- Bi\\u1ec3n s\\u1ed1 P.103a: C\\u1ea5m xe \\u00f4 t\\u00f4;\\n- Bi\\u1ec3n s\\u1ed1 P.103(b,c): C\\u1ea5m xe \\u00f4 t\\u00f4 r\\u1ebd tr\\u00e1i; C\\u1ea5m xe \\u00f4t\\u00f4 r\\u1ebd ph\\u1ea3i